# Лабораторная работа №4: KNN и K-means

## Классификация и кластеризация

---

## 0. О лабораторной работе

### Цели работы
1. Изучить алгоритм **k-ближайших соседей (KNN)** для классификации
2. Изучить алгоритм **K-means** для кластеризации
3. Понять разницу между supervised и unsupervised обучением
4. Научиться считать расстояния между точками вручную

### Структура работы
- **Часть 1:** KNN — классификация ирисов с разными метриками расстояния
- **Часть 2:** K-means — кластеризация синтетических данных

---

## ЧАСТЬ 1: KNN (k-Nearest Neighbors)

### Теоретический блок

#### Идея алгоритма

**KNN** — «Скажи мне, кто твои соседи, и я скажу, кто ты».

Для классификации новой точки:
1. Найти **k** ближайших соседей в обучающей выборке
2. Посмотреть их классы
3. Назначить класс, который встречается чаще всего (голосование)

---

#### Метрики расстояния

**Евклидово расстояние** (прямая линия):

$$d(x, y) = \sqrt{\sum_{i=1}^n (x_i - y_i)^2}$$

**Манхэттенское расстояние** (сумма абсолютных разностей):

$$d(x, y) = \sum_{i=1}^n |x_i - y_i|$$

**Косинусное расстояние**:

$$d(x, y) = 1 - \frac{x \cdot y}{||x|| \cdot ||y||}$$

---

#### Выбор параметра k

| k | Эффект |
|---|--------|
| **k=1** | Слишком чувствительно к шуму (overfitting) |
| **k=нечётное** | Избегает ничьих при бинарной классификации |
| **k=большое** | Слишком сглаживает границы (underfitting) |

**Правило:** Выбирайте нечётное k для бинарной классификации!

---

#### Почему важна нормализация?

Представьте данные: `рост (см)` и `доход (руб)`.

- Рост: 170-190 см (разница ~20)
- Доход: 30000-100000 руб (разница ~70000)

**Без нормализации** доход будет доминировать в расстоянии!

---

In [ ]:
# Импорт библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Библиотеки загружены!")

---

## 1. Определение варианта

### Система вариантов

| Вариант | Метрика расстояния |
|---------|-------------------|
| 1, 4, 7, 10, 13, 16, 19 | **Евклидово** |
| 2, 5, 8, 11, 14, 17, 20 | **Манхэттенское** |
| 3, 6, 9, 12, 15, 18 | **Косинусное** |

**Формула:** `ID = (№ по списку - 1) % 20 + 1`

In [ ]:
# ============================================
# ВАЖНО: Замените на ваш номер по списку!
# ============================================
NOMER_V_SPISKE = 1  # ← Замените на ваш номер

# Вычисление ID варианта (НЕ ИЗМЕНЯЙТЕ)
ID = (NOMER_V_SPISKE - 1) % 20 + 1

# Определение метрики по варианту
variant_group = (ID - 1) % 3
if variant_group == 0:
    METRIC = 'euclidean'  # Варианты 1, 4, 7, ...
    METRIC_NAME = 'Евклидово расстояние'
elif variant_group == 1:
    METRIC = 'manhattan'   # Варианты 2, 5, 8, ...
    METRIC_NAME = 'Манхэттенское расстояние'
else:
    METRIC = 'cosine'      # Варианты 3, 6, 9, ...
    METRIC_NAME = 'Косинусное расстояние'

print(f"ID варианта: {ID}")
print(f"Метрика: {METRIC_NAME}")

---

## 2. Датасет Iris

### Описание

Датасет содержит измерения **150 ирисов** трёх видов:

| Класс | Название |
|-------|----------|
| 0 | Setosa (щетинистый) |
| 1 | Versicolor (разноцветный) |
| 2 | Virginica (виргинский) |

**Признаки:**
- `sepal_length` — длина чашелистика (см)
- `sepal_width` — ширина чашелистика (см)
- `petal_length` — длина лепестка (см)
- `petal_width` — ширина лепестка (см)

In [ ]:
# Загрузка датасета Iris
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df['species'] = [target_names[i] for i in y]

print(f"Размер датасета: {df.shape[0]} цветков")
print(f"\nКлассы: {target_names}")
print(f"\nПризнаки: {feature_names}")
df.head(10)

In [ ]:
# Визуализация: пары признаков
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый график: sepal
for species, color in zip(target_names, ['red', 'green', 'blue']):
    mask = df['species'] == species
    axes[0].scatter(df.loc[mask, 'sepal length (cm)'], 
                   df.loc[mask, 'sepal width (cm)'],
                   c=color, label=species, alpha=0.7, s=50)
axes[0].set_xlabel('Sepal length (cm)')
axes[0].set_ylabel('Sepal width (cm)')
axes[0].set_title('Sepal dimensions')
axes[0].legend()

# Правый график: petal
for species, color in zip(target_names, ['red', 'green', 'blue']):
    mask = df['species'] == species
    axes[1].scatter(df.loc[mask, 'petal length (cm)'], 
                   df.loc[mask, 'petal width (cm)'],
                   c=color, label=species, alpha=0.7, s=50)
axes[1].set_xlabel('Petal length (cm)')
axes[1].set_ylabel('Petal width (cm)')
axes[1].set_title('Petal dimensions')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nНаблюдение: по petal-признакам классы разделяются лучше!")

---

## 3. Задание: Ручной расчёт расстояний

### 3.1. Подвыборка для расчётов

Вам дана подвыборка из 5 точек обучающей выборки и 1 новая точка.

**Ваша задача:**
1. Рассчитать расстояния от новой точки до всех 5 точек обучающей выборки
2. Использовать вашу метрику варианта

In [ ]:
# Генерация подвыборки (seed = ID варианта)
np.random.seed(ID)

# Выбираем 5 случайных точек из Iris
train_indices = np.random.choice(len(X), 5, replace=False)
X_train_small = X[train_indices]
y_train_small = y[train_indices]

# Создаём новую точку (близко к центру датасета + случайное смещение)
new_point = np.array([5.8, 3.0, 4.2, 1.3]) + np.random.randn(4) * 0.2

# Таблица с данными
print("=" * 70)
print(f"ВАРИАНТ {ID}: {METRIC_NAME}")
print("=" * 70)
print("\nОБУЧАЮЩАЯ ВЫБОРКА (5 точек):\n")

train_df = pd.DataFrame(X_train_small, columns=feature_names)
train_df['Класс'] = [target_names[i] for i in y_train_small]
train_df.index.name = '№'
print(train_df.to_string())

print("\n" + "=" * 70)
print("НОВАЯ ТОЧКА (для классификации):\n")
new_df = pd.DataFrame([new_point], columns=feature_names)
new_df.index = ['Новая точка']
print(new_df.to_string())
print("\n" + "=" * 70)

### 3.2. Формулы для расчёта

**Для вашей метрики:**

In [ ]:
# Вывод формулы для вашего варианта
print(f"Ваша метрика: {METRIC_NAME}\n")

if METRIC == "euclidean":
    print("Формула Евклидова расстояния:")
    print("d(x, y) = sqrt((x1-y1)² + (x2-y2)² + (x3-y3)² + (x4-y4)²)\n")
    print("Пример расчёта для точки 1:")
    print(f"  Новая точка: ({new_point[0]:.2f}, {new_point[1]:.2f}, {new_point[2]:.2f}, {new_point[3]:.2f})")
    print(
        f"  Точка 1:     ({X_train_small[0][0]:.2f}, {X_train_small[0][1]:.2f}, {X_train_small[0][2]:.2f}, {X_train_small[0][3]:.2f})"
    )
    diff = new_point - X_train_small[0]
    print(f"  Разности:    ({diff[0]:.2f}, {diff[1]:.2f}, {diff[2]:.2f}, {diff[3]:.2f})")
    print(f"  Квадраты:    ({diff[0] ** 2:.2f}, {diff[1] ** 2:.2f}, {diff[2] ** 2:.2f}, {diff[3] ** 2:.2f})")
    print(f"  Сумма:       {np.sum(diff**2):.2f}")
    print(f"  Расстояние:  {np.sqrt(np.sum(diff**2)):.4f}")

elif METRIC == "manhattan":
    print("Формула Манхэттенского расстояния:")
    print("d(x, y) = |x1-y1| + |x2-y2| + |x3-y3| + |x4-y4|\n")
    print("Пример расчёта для точки 1:")
    print(f"  Новая точка: ({new_point[0]:.2f}, {new_point[1]:.2f}, {new_point[2]:.2f}, {new_point[3]:.2f})")
    print(
        f"  Точка 1:     ({X_train_small[0][0]:.2f}, {X_train_small[0][1]:.2f}, {X_train_small[0][2]:.2f}, {X_train_small[0][3]:.2f})"
    )
    diff = np.abs(new_point - X_train_small[0])
    print(f"  |Разности|:  ({diff[0]:.2f}, {diff[1]:.2f}, {diff[2]:.2f}, {diff[3]:.2f})")
    print(f"  Сумма:       {np.sum(diff):.4f}")

else:  # cosine
    print("Формула косинусного расстояния:")
    print("d(x, y) = 1 - (x·y) / (||x|| * ||y||)\n")
    print("Пример расчёта для точки 1:")
    dot_product = np.dot(new_point, X_train_small[0])
    norm_new = np.linalg.norm(new_point)
    norm_train = np.linalg.norm(X_train_small[0])
    cosine_sim = dot_product / (norm_new * norm_train)
    cosine_dist = 1 - cosine_sim
    print(f"  x·y (скалярное произведение):    {dot_product:.2f}")
    print(f"  ||x|| (длина новой точки):      {norm_new:.4f}")
    print(f"  ||y|| (длина точки 1):          {norm_train:.4f}")
    print(f"  Косинус сходства:               {cosine_sim:.4f}")
    print(f"  Косинус расстояние:             {cosine_dist:.4f}")

---

### 3.3. Таблица для заполнения

**Рассчитайте расстояния вручную и заполните таблицу:**

| Точка | sepal_l | sepal_w | petal_l | petal_w | Класс | **Расстояние** |
|-------|---------|---------|---------|---------|-------|----------------|
| 1 | | | | | | **____** |
| 2 | | | | | | **____** |
| 3 | | | | | | **____** |
| 4 | | | | | | **____** |
| 5 | | | | | | **____** |

*Подсказка: значения признаков возьмите из таблицы выше*

---

## 4. Проверка ручного расчёта

После заполнения таблицы запустите ячейку ниже для проверки.

In [ ]:
# ============================================
# ПРОВЕРКА: Расстояния (запустите после расчёта)
# ============================================

# Расчёт расстояний вручную (numpy)
def calculate_distances(new_point, train_points, metric):
    """Вычисляет расстояния от new_point до всех train_points"""
    distances = []
    for point in train_points:
        if metric == 'euclidean':
            d = np.sqrt(np.sum((new_point - point) ** 2))
        elif metric == 'manhattan':
            d = np.sum(np.abs(new_point - point))
        else:  # cosine
            dot = np.dot(new_point, point)
            norm_new = np.linalg.norm(new_point)
            norm_point = np.linalg.norm(point)
            d = 1 - (dot / (norm_new * norm_point))
        distances.append(d)
    return np.array(distances)

distances = calculate_distances(new_point, X_train_small, METRIC)

# Таблица с результатами
result_df = pd.DataFrame({
    'Точка': range(1, 6),
    'Класс': [target_names[i] for i in y_train_small],
    f'{METRIC_NAME}': distances
})

print("\n" + "="*70)
print("ТАБЛИЦА РАССТОЯНИЙ (для проверки):\n")
print(result_df.to_string(index=False))
print("="*70)

# Сортировка по расстоянию
sorted_idx = np.argsort(distances)
print("\nТОЧКИ В ПОРЯДКЕ УДАЛЕНИЯ (от ближайшей к самой далёкой):\n")
for rank, idx in enumerate(sorted_idx, 1):
    print(f"{rank}. Точка {idx+1}: {distances[idx]:.4f} → Класс: {target_names[y_train_small[idx]]}")

In [ ]:
# Визуализация расстояний
plt.figure(figsize=(10, 5))
bars = plt.bar(range(1, 6), distances, color=["#1a2bdf" if i == sorted_idx[0] else "#35fba5" for i in range(5)])
plt.xlabel('Номер точки')
plt.ylabel(f'{METRIC_NAME}')
plt.title(f'Расстояния от новой точки до обучающей выборки\n(синий = ближайший сосед)')
plt.xticks(range(1, 6))
plt.grid(axis='y', alpha=0.3)

# Добавляем значения на бары
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.4f}',
             ha='center', va='bottom', fontsize=10)

plt.show()

print("\nВопрос: Какие классы у 3 ближайших соседей?")

---

## 5. Предсказание класса

### 5.1. Ручное предсказание для k=3

**Ваша задача:**
1. Выберите **3 ближайших соседа** (из таблицы расстояний)
2. Запишите их классы
3. Сделайте предсказание путём голосования

**Таблица для заполнения:**

| Ранг | Точка | Расстояние | Класс |
|------|-------|------------|-------|
| 1 (ближайший) | | | |
| 2 | | | |
| 3 | | | |

**Голосование:**
- Класс 0 (Setosa): ____ голосов
- Класс 1 (Versicolor): ____ голосов
- Класс 2 (Virginica): ____ голосов

**Предсказание для k=3:** ____

### 5.2. Проверка предсказания

In [ ]:
# ============================================
# ПРОВЕРКА: Предсказание для разных k
# ============================================

print("=" * 70)
print(f"ПРЕДСКАЗАНИЯ ДЛЯ НОВОЙ ТОЧКИ (метрика: {METRIC_NAME})\n")

for k in [1, 3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k, metric=METRIC)
    knn.fit(X_train_small, y_train_small)
    pred = knn.predict([new_point])[0]

    # Соседи
    distances, indices = knn.kneighbors([new_point])
    neighbor_classes = y_train_small[indices[0]]

    print(f"k = {k}:")
    print(f"  Ближайшие соседи: {[target_names[i] for i in neighbor_classes]}")
    print(f"  Предсказанный класс: {target_names[pred]}")
    print()

print("=" * 70)

# Финальное предсказание для k=3
knn3 = KNeighborsClassifier(n_neighbors=3, metric=METRIC)
knn3.fit(X_train_small, y_train_small)
final_pred = knn3.predict([new_point])[0]
print(f"\nФИНАЛЬНОЕ ПРЕДСКАЗАНИЕ (k=3): {target_names[final_pred]}")

---

## 6. Исследование параметра k

### Как k влияет на точность классификации?

In [ ]:
# Разделение полного датасета
# Используем stratify для баланса классов
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123, stratify=y
)

print(f"Train: {len(X_train)} точек")
print(f"Test: {len(X_test)} точек")
print(f"\nРаспределение классов в test:")
for i, name in enumerate(target_names):
    print(f"  {name}: {sum(y_test == i)}")

In [ ]:
# Исследование разных k
k_values = range(1, 31, 2)
train_acc = []
test_acc = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric=METRIC)
    knn.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, knn.predict(X_train)))
    test_acc.append(accuracy_score(y_test, knn.predict(X_test)))

# Вывод таблицы
print(f"\nВАРИАНТ {ID}: {METRIC_NAME}")
print("\n   k   | Train Acc | Test Acc | Разница")
print("-" * 40)
for k, tr, te in zip(k_values, train_acc, test_acc):
    print(f"  {k:2d}   |   {tr:.3f}   |  {te:.3f}  |  {tr-te:+.3f}")

In [ ]:
# График зависимости accuracy от k
plt.figure(figsize=(10, 5))
plt.plot(k_values, train_acc, 'b-o', label='Train Accuracy', linewidth=2, markersize=6)
plt.plot(k_values, test_acc, 'r-s', label='Test Accuracy', linewidth=2, markersize=6)
plt.xlabel('k (количество соседей)')
plt.ylabel('Accuracy')
plt.title(f'Зависимость точности от k (метрика: {METRIC_NAME})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(k_values)

# Лучшее k
best_k_idx = np.argmax(test_acc)
best_k = k_values[best_k_idx]
best_acc = test_acc[best_k_idx]
plt.axvline(x=best_k, color='green', linestyle='--', alpha=0.5, label=f'Лучшее k={best_k}')
plt.legend()

plt.show()

print(f"\nЛучшее k = {best_k} с Test Accuracy = {best_acc:.3f}")

### Вопросы для обсуждения:

1. **Что происходит при k=1?** Почему train accuracy = 1.0?

```
Ваш ответ:
...
```

2. **Почему test accuracy падает при очень большом k?**

```
Ваш ответ:
...
```

3. **Какое оптимальное k для вашего варианта?**

```
Ответ: k = ____
```

---

## 7. Визуализация границ принятия решений

Как KNN разделяет пространство признаков?

In [ ]:
# Визуализация границ решений (на 2 признаках: petal_length, petal_width)
from matplotlib.colors import ListedColormap

# Берём только 2 признака
X_2d = X[:, [2, 3]]  # petal_length, petal_width

# Обучаем KNN с оптимальным k
knn_vis = KNeighborsClassifier(n_neighbors=best_k, metric=METRIC)
knn_vis.fit(X_2d, y)

# Создаём сетку для визуализации
x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

# Предсказываем для каждой точки сетки
Z = knn_vis.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Визуализация
plt.figure(figsize=(10, 6))
cmap_light = ListedColormap(['#FFB6C1', '#90EE90', '#ADD8E6'])
cmap_bold = ['red', 'green', 'blue']

plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.4)
for i, color in enumerate(cmap_bold):
    plt.scatter(X_2d[y == i, 0], X_2d[y == i, 1], 
                c=color, label=target_names[i], 
                edgecolors='black', alpha=0.7)

plt.xlabel('Petal length (cm)')
plt.ylabel('Petal width (cm)')
plt.title(f'Границы решений KNN (k={best_k}, метрика: {METRIC_NAME})')
plt.legend()
plt.show()

print("\nНаблюдение: границы решений KNN - это многоугольники Вороного!")

---

---

# ЧАСТЬ 2: K-means (Кластеризация)

### Теоретический блок

#### Идея алгоритма

**K-means** — это алгоритм **unsupervised learning** (обучение без учителя).

Он НЕ знает классы объектов. Его цель — **сгруппировать похожие объекты вместе**.

---

#### Алгоритм пошагово:

1. **Инициализация:** Выбрать k случайных центроидов (центров кластеров)

2. **Назначение:** Для каждой точки найти ближайший центроид и отнести точку к этому кластеру

3. **Обновление:** Пересчитать центроиды как среднее всех точек в кластере

4. **Повтор:** Шаги 2-3 пока центроиды не перестанут меняться

---

#### Формула обновления центроиды:

$$\mu_j = \frac{1}{n_j} \sum_{x \in C_j} x$$

Где $\mu_j$ — новый центроид кластера $j$, $n_j$ — количество точек в кластере.

---

#### Метод локтя (Elbow Method)

Как выбрать оптимальное k?

- **Inertia** = сумма квадратов расстояний от точек до их центроидов
- При k=1: Inertia большая
- При k=N: Inertia = 0 (каждая точка — свой кластер)
- **Оптимальное k**: точка «излома» на графике

---

## 8. Интерактивная практика K-means

### 8.1. Генерация данных

Сгенерируем синтетические данные для кластеризации (seed = ID варианта).

In [ ]:
# ============================================
# ГЕНЕРАЦИЯ ДАННЫХ (seed = ID варианта)
# ============================================
np.random.seed(ID)

# Генерируем 3 кластера в 2D (более разделённые)
n_per_cluster = 30

# Кластер 1 (левый нижний)
cluster1 = np.random.randn(n_per_cluster, 2) * 0.8 + [2, 2]
# Кластер 2 (правый нижний)  
cluster2 = np.random.randn(n_per_cluster, 2) * 0.8 + [8, 2]
# Кластер 3 (центральный верхний)
cluster3 = np.random.randn(n_per_cluster, 2) * 0.8 + [5, 8]

# Объединяем
X_kmeans = np.vstack([cluster1, cluster2, cluster3])

print(f"Сгенерировано {len(X_kmeans)} точек для кластеризации")
print(f"Seed генерации: {ID}\n")
print("Первые 10 точек:")
print(pd.DataFrame(X_kmeans, columns=['x', 'y']).head(10).to_string(index=False))

In [ ]:
# Визуализация данных (без меток классов!)
plt.figure(figsize=(8, 8))
plt.scatter(X_kmeans[:, 0], X_kmeans[:, 1], c='gray', alpha=0.6, s=50, edgecolors='black')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Данные для кластеризации (seed={ID})')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='k', linewidth=0.5)
plt.axvline(x=0, color='k', linewidth=0.5)
plt.show()

print("\nВопрос: На сколько кластеров визуально делятся эти точки?")

### 8.2. Инициализация центроидов

Задаём k=3 и инициализируем случайные центроиды.

In [ ]:
# Инициализация случайных центроидов
k = 3
np.random.seed(ID)  # Фиксируем seed для воспроизводимости

initial_centroids = X_kmeans[np.random.choice(len(X_kmeans), k, replace=False)]

print(f"k = {k}")
print("\nНАЧАЛЬНЫЕ ЦЕНТРОИДЫ:\n")
for i, centroid in enumerate(initial_centroids):
    print(f"Центроид {i+1}: ({centroid[0]:.2f}, {centroid[1]:.2f})")

# Визуализация
plt.figure(figsize=(8, 8))
plt.scatter(X_kmeans[:, 0], X_kmeans[:, 1], c='gray', alpha=0.5, s=50, label='Точки данных')
colors = ['red', 'blue', 'green']
markers = ['X', 'X', 'X']
for i, centroid in enumerate(initial_centroids):
    plt.scatter(centroid[0], centroid[1], c=colors[i], s=300, 
                marker=markers[i], edgecolors='black', linewidth=2,
                label=f'Центроид {i+1}')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Итерация 0: Начальные центроиды (seed={ID})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

## 9. Итерация 1: Ручной расчёт

### 9.1. Шаг назначения (Assignment)

**Ваша задача:** Для каждой точки данных определить ближайший центроид.

Используйте **Евклидово расстояние** (стандарт для K-means):

$$d = \sqrt{(x - \mu_x)^2 + (y - \mu_y)^2}$$

**Таблица для заполнения (первые 5 точек):**

| Точка | x | y | dist до C1 | dist до C2 | dist до C3 | **Кластер** |
|-------|---|---|------------|------------|------------|-------------|
| 1 | | | | | | **_** |
| 2 | | | | | | **_** |
| 3 | | | | | | **_** |
| 4 | | | | | | **_** |
| 5 | | | | | | **_** |

*Центроиды: возьмите из вывода выше*

### 9.2. Шаг обновления (Update)

**Ваша задача:** Рассчитать новые координаты центроидов.

**Формула:** Новый центроид = среднее всех точек в кластере

$$\mu_{new} = \left(\frac{\sum x}{n}, \frac{\sum y}{n}\right)$$

**Таблица для заполнения:**

| Кластер | Кол-во точек | Σx | Σy | Новый μx | Новый μy |
|---------|--------------|----|----|----------|----------|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Новые центроиды:**
- C1: (__, __)
- C2: (__, __)
- C3: (__, __)

---

## 10. Проверка итерации 1

**После ручного расчёта заполните новые центроиды в код ниже и запустите для проверки.**

In [ ]:
# ============================================
# ВАЖНО: Заполните новые центроиды после ручного расчёта!
# ============================================

# Уберите комментарий и заполните ВАШИ значения:
# NEW_CENTROIDS = np.array([
#     [____, ____],  # Центроид 1 (новый)
#     [____, ____],  # Центроид 2 (новый)
#     [____, ____],  # Центроид 3 (новый)
# ])

# Для демонстрации используем автоматический расчёт
# (уберите это после заполнения вручную)
from sklearn.cluster import KMeans
kmeans_check = KMeans(n_clusters=3, init=initial_centroids, n_init=1, max_iter=1, random_state=ID)
kmeans_check.fit(X_kmeans)
NEW_CENTROIDS = kmeans_check.cluster_centers_

print("Новые центроиды:")
for i, c in enumerate(NEW_CENTROIDS):
    print(f"C{i+1}: ({c[0]:.2f}, {c[1]:.2f})")

In [ ]:
# ============================================
# ПРОВЕРКА: Итерация 1 (запустите после заполнения)
# ============================================

# Правильный расчёт с sklearn
kmeans_iter1 = KMeans(n_clusters=3, init=initial_centroids, n_init=1, max_iter=1, random_state=ID)
kmeans_iter1.fit(X_kmeans)

correct_centroids = kmeans_iter1.cluster_centers_
correct_labels = kmeans_iter1.labels_

print("="*60)
print("ПРОВЕРКА ИТЕРАЦИИ 1\n")
print("ПРАВИЛЬНЫЕ новые центроиды:")
for i, c in enumerate(correct_centroids):
    print(f"C{i+1}: ({c[0]:.2f}, {c[1]:.2f})")

# Сравнение
print("\nРАЗНИЦА с вашими значениями:")
for i, (yours, correct) in enumerate(zip(NEW_CENTROIDS, correct_centroids)):
    diff = np.abs(yours - correct)
    print(f"C{i+1}: {diff[0]:.3f}, {diff[1]:.3f}", end="")
    if np.all(diff < 0.1):
        print(" ✓ Совпало!")
    else:
        print(" ✗ Есть расхождение")

print("="*60)

In [ ]:
# Визуализация после итерации 1
plt.figure(figsize=(12, 5))

# Левый график: назначение кластеров
plt.subplot(1, 2, 1)
colors_map = ['red', 'blue', 'green']
for i in range(3):
    mask = correct_labels == i
    plt.scatter(X_kmeans[mask, 0], X_kmeans[mask, 1], 
                c=colors_map[i], alpha=0.5, s=50, label=f'Кластер {i+1}')
for i, centroid in enumerate(initial_centroids):
    plt.scatter(centroid[0], centroid[1], c=colors_map[i], s=200, 
                marker='X', edgecolors='black', linewidth=2)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Итерация 1: Назначение кластеров\n(старые центроиды)')
plt.legend()
plt.grid(True, alpha=0.3)

# Правый график: новые центроиды
plt.subplot(1, 2, 2)
for i in range(3):
    mask = correct_labels == i
    plt.scatter(X_kmeans[mask, 0], X_kmeans[mask, 1], 
                c=colors_map[i], alpha=0.5, s=50, label=f'Кластер {i+1}')
# Стрелки от старых к новым центроидам
for i, (old, new) in enumerate(zip(initial_centroids, correct_centroids)):
    plt.scatter(old[0], old[1], c=colors_map[i], s=150, marker='o', 
                edgecolors='black', linewidth=1, alpha=0.5)
    plt.scatter(new[0], new[1], c=colors_map[i], s=200, marker='X',
                edgecolors='black', linewidth=2)
    plt.arrow(old[0], old[1], new[0]-old[0], new[1]-old[1],
              head_width=0.2, head_length=0.2, fc=colors_map[i], ec=colors_map[i], alpha=0.5)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Итерация 1: Новые центроиды\n(стрелки = движение)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 11. Итерация 2

### 11.1. Новый шаг назначения

Используйте **новые центроиды** из проверки выше.

**Вопрос:** Изменились ли назначения кластеров по сравнению с итерацией 1?

```
Ответ:
...
```

### 11.2. Новые центроиды после итерации 2

**Рассчитайте новые центроиды:**

| Кластер | Новый μx | Новый μy |
|---------|----------|----------|
| 1 | | |
| 2 | | |
| 3 | | |

---

## 12. Проверка итерации 2

In [ ]:
# ============================================
# ПРОВЕРКА: Итерация 2
# ============================================

# Запускаем K-means на 2 итерации
kmeans_iter2 = KMeans(n_clusters=3, init=initial_centroids, n_init=1, max_iter=2, random_state=ID)
kmeans_iter2.fit(X_kmeans)

print("="*60)
print("ИТЕРАЦИЯ 2\n")
print("Центроиды после итерации 2:")
for i, c in enumerate(kmeans_iter2.cluster_centers_):
    print(f"C{i+1}: ({c[0]:.2f}, {c[1]:.2f})")

# Сравнение с итерацией 1
print("\nИзменение центроидов (итерация 2 → итерация 1):")
for i, (c2, c1) in enumerate(zip(kmeans_iter2.cluster_centers_, correct_centroids)):
    diff = np.abs(c2 - c1)
    print(f"C{i+1}: {diff[0]:.4f}, {diff[1]:.4f}")

print("="*60)

In [ ]:
# Визуализация движения центроидов
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Итерация 0
axes[0].scatter(X_kmeans[:, 0], X_kmeans[:, 1], c='gray', alpha=0.3)
for i, c in enumerate(initial_centroids):
    axes[0].scatter(c[0], c[1], c=['red', 'blue', 'green'][i], s=200, marker='X', edgecolors='black')
axes[0].set_title('Итерация 0\n(Начальные центроиды)')
axes[0].grid(True, alpha=0.3)

# Итерация 1
axes[1].scatter(X_kmeans[:, 0], X_kmeans[:, 1], c='gray', alpha=0.3)
for i, (old, new) in enumerate(zip(initial_centroids, correct_centroids)):
    axes[1].scatter(old[0], old[1], c=['red', 'blue', 'green'][i], s=100, marker='o', alpha=0.3)
    axes[1].scatter(new[0], new[1], c=['red', 'blue', 'green'][i], s=200, marker='X', edgecolors='black')
    axes[1].arrow(old[0], old[1], new[0]-old[0], new[1]-old[1],
                  head_width=0.2, head_length=0.2, fc=['red', 'blue', 'green'][i], ec=['red', 'blue', 'green'][i], alpha=0.5)
axes[1].set_title('Итерация 1')
axes[1].grid(True, alpha=0.3)

# Итерация 2
axes[2].scatter(X_kmeans[:, 0], X_kmeans[:, 1], c='gray', alpha=0.3)
for i, (old, new) in enumerate(zip(correct_centroids, kmeans_iter2.cluster_centers_)):
    axes[2].scatter(old[0], old[1], c=['red', 'blue', 'green'][i], s=100, marker='o', alpha=0.3)
    axes[2].scatter(new[0], new[1], c=['red', 'blue', 'green'][i], s=200, marker='X', edgecolors='black')
    axes[2].arrow(old[0], old[1], new[0]-old[0], new[1]-old[1],
                  head_width=0.2, head_length=0.2, fc=['red', 'blue', 'green'][i], ec=['red', 'blue', 'green'][i], alpha=0.5)
axes[2].set_title('Итерация 2')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 13. Сходимость алгоритма

Когда K-means останавливается?

In [ ]:
# Полный K-means до сходимости
kmeans_final = KMeans(n_clusters=3, init=initial_centroids, n_init=1, random_state=ID)
kmeans_final.fit(X_kmeans)

print(f"Количество итераций до сходимости: {kmeans_final.n_iter_}")
print(f"\nФинальные центроиды:")
for i, c in enumerate(kmeans_final.cluster_centers_):
    print(f"C{i+1}: ({c[0]:.2f}, {c[1]:.2f})")
print(f"\nInertia (сумма расстояний до центроидов): {kmeans_final.inertia_:.2f}")

In [ ]:
# Финальная визуализация
plt.figure(figsize=(10, 8))

colors_map = ['red', 'blue', 'green']
for i in range(3):
    mask = kmeans_final.labels_ == i
    plt.scatter(X_kmeans[mask, 0], X_kmeans[mask, 1], 
                c=colors_map[i], alpha=0.6, s=50, label=f'Кластер {i+1}', edgecolors='black')

# Центроиды
for i, centroid in enumerate(kmeans_final.cluster_centers_):
    plt.scatter(centroid[0], centroid[1], c=colors_map[i], s=400, 
                marker='X', edgecolors='black', linewidth=3)

plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Финальная кластеризация (k=3, итераций: {kmeans_final.n_iter_})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

## 14. Метод локтя (Elbow Method)

Как выбрать оптимальное k?

In [ ]:
# K-means для разных k
k_range = range(1, 11)
inertias = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=ID)
    kmeans.fit(X_kmeans)
    inertias.append(kmeans.inertia_)

# Таблица
print("\n   k   |  Inertia")
print("-" * 20)
for k, inertia in zip(k_range, inertias):
    print(f"  {k:2d}   |  {inertia:.2f}")

In [ ]:
# График метода локтя
plt.figure(figsize=(10, 5))
plt.plot(k_range, inertias, 'b-o', linewidth=2, markersize=8)
plt.xlabel('k (количество кластеров)')
plt.ylabel('Inertia')
plt.title('Метод локтя (Elbow Method)')
plt.grid(True, alpha=0.3)
plt.xticks(k_range)

# Метод локтя: ищем точку максимального изгиба
# Вычисляем относительное уменьшение inertia
diffs = np.diff(inertias)
rel_diffs = -diffs / inertias[:-1]  # относительное уменьшение в %

# Находим k после которого уменьшение становится < 20%
threshold = 0.20
elbow_k = 3  # по умолчанию
for i, rel in enumerate(rel_diffs):
    if rel < threshold:
        elbow_k = k_range[i]
        break

plt.axvline(x=elbow_k, color='red', linestyle='--', 
            label=f'Локоть при k={elbow_k}', alpha=0.7)
plt.legend()

plt.show()

print(f"\nРекомендуемое k (по методу локтя): k = {elbow_k}")
print("\nПояснение: локоть - это точка, после которой")
print("увеличение k даёт небольшое уменьшение inertia.")

### Вопрос:

**Почему мы выбираем k=3, а не k=10 (даёт Inertia=0)?**

```
Ваш ответ:
...
```

---

## 15. Итоговые вопросы

### Разница между KNN и K-means

**Вопрос 1:** В чём главное отличие между KNN и K-means?

- KNN: supervised / unsupervised?
- K-means: supervised / unsupervised?

```
Ответ:
...
```

**Вопрос 2:** Почему для KNN важна нормализация признаков, а для деревьев решений — нет?

```
Ответ:
...
```

**Вопрос 3:** Что произойдёт, если выбрать k=1 в KNN? А k=N (равно размеру выборки)?

```
Ответ:
...
```

**Вопрос 4:** Как выбрать оптимальное k в K-means, если метод локтя не даёт явного излома?

```
Ответ:
...
```

**Вопрос 5:** Может ли K-means найти «плохие» кластеры? Приведите пример ситуации.

```
Ответ:
...
```

**Вопрос 6:** Зависит ли результат K-means от начальной инициализации центроидов?

```
Ответ:
...
```

**Вопрос 7:** Какая метрика расстояния используется в K-means по умолчанию? Можно ли использовать манхэттенское расстояние?

```
Ответ:
...
```

**Вопрос 8:** В чём преимущество KNN перед сложными моделями (нейросети, бустинг)?

```
Ответ:
...
```